[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Mach_Learn/Training_Dynamics.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Training Dynamics

The craft between [ANN's](./Intro_ANN/Intro_ANN.ipynb) backprop and [Scale_NN's](./Scale_NN/Scale_NN.ipynb) systems view: which optimizer, what schedule, how much regularization — with every claim demonstrated by an experiment you can rerun.

## 1. Pre-requisites

- [Intro to ANN](./Intro_ANN/Intro_ANN.ipynb) and [Intro to PyTorch](../Intro_DL_4_Physics/intro_pytorch/intro_pytorch.ipynb).
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) — condition numbers and SGD noise floors.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

# Standard testbed for the whole workshop: noisy spiral classification (from the ANN workshop)
def spirals(n=1500, noise=0.35, seed=0):
    r = np.random.default_rng(seed)
    t = np.linspace(0.5, 3*np.pi, n//2)
    X, y = [], []
    for cls, ph in [(0, 0.0), (1, np.pi)]:
        X.append(np.stack([t*np.cos(t+ph), t*np.sin(t+ph)], 1) + noise*r.standard_normal((n//2, 2)))
        y.append(np.full(n//2, cls))
    X = np.concatenate(X).astype(np.float32); y = np.concatenate(y)
    X = (X - X.mean(0)) / X.std(0)
    idx = r.permutation(n)
    return map(torch.from_numpy, (X[idx][:1000], y[idx][:1000].astype(np.int64),
                                  X[idx][1000:], y[idx][1000:].astype(np.int64)))
Xtr, ytr, Xte, yte = spirals()

def make_model(width=64):
    torch.manual_seed(1)
    return nn.Sequential(nn.Linear(2, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 2))

def train(model, opt, epochs=150, sched=None, wd_manual=0.0):
    lossf = nn.CrossEntropyLoss(); hist = []
    for ep in range(epochs):
        for i in range(0, 1000, 100):
            xb, yb = Xtr[i:i+100], ytr[i:i+100]
            opt.zero_grad(); loss = lossf(model(xb), yb); loss.backward(); opt.step()
        if sched: sched.step()
        with torch.no_grad():
            hist.append(( lossf(model(Xtr), ytr).item(),
                          (model(Xte).argmax(1) == yte).float().mean().item() ))
    return np.array(hist)

---
### 🕐 Session 1 of 3 — *Optimizers: SGD → Momentum → Adam* (~40 min)
**Goal:** understand what each optimizer adds, and race them fairly.
**Builds on:** [Optimization](../Intro_Math/Optimization/Optimization.ipynb) S2/S4. &nbsp; **Feeds into:** Session 2 (schedules).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: Optimizers: SGD → Momentum → Adam</b></summary>

**Timing (~40 min).** 10 min what momentum adds · 12 min what Adam adds · 10 min the race · 8 min reading the result honestly.

**Frame the workshop's position in the curriculum first.** [ANN](./Intro_ANN/Intro_ANN.ipynb) gave the gradient, [Scale_NN](./Scale_NN/Scale_NN.ipynb) gives the systems view, and this workshop is the craft in between: *which* optimiser, *what* schedule, *how much* regularisation. The distinguishing feature is that every claim here is demonstrated by an experiment you can rerun — which also means every claim is subject to the honest question of whether the experiment supports it.

**Teach momentum as a physics correction, not a formula.** Plain SGD treats the gradient as a *velocity*: where the slope points, you go. Momentum treats it as a **force**: it changes your velocity, and velocity persists. The consequence is the one that matters — in a narrow canyon, the across-canyon components alternate sign and cancel on accumulation, while the along-canyon component is consistent and builds. **Zigzag cancels, progress accumulates.** Point straight back at the κ-canyon picture from [Optimization](../Intro_Math/Optimization/Optimization.ipynb) S2; that plot is the reason momentum exists.

**Give the effective-step-size number, because it makes the hyperparameter concrete.** With momentum $\beta$, a persistent gradient produces a steady-state step of $\frac{1}{1-\beta}$ times the SGD step — so $\beta = 0.9$ is a **10× amplification** along consistent directions. That single arithmetic fact explains both why momentum accelerates so dramatically and why a learning rate tuned for plain SGD often diverges when momentum is switched on. Students who know it stop treating $\beta$ as a magic 0.9.

**Then Adam, and resist letting it sound like magic.** It maintains a per-coordinate running RMS of recent gradients and divides each step by it. Coordinates that rarely receive gradient get large steps; jittery ones get small steps. **This is approximate per-axis preconditioning** — a diagonal approximation to the inverse Hessian's scale, which is exactly the κ problem from the Optimization workshop attacked coordinate-wise. Neither optimiser knows anything about your data; both attack curvature and scale imbalance and nothing else.

**Be explicit about the comparison problem before running the race, because it is a methodological trap the room should learn to spot.** SGD is run at lr 0.05 and Adam at 1e-3, and those numbers are not comparable — Adam's normalised updates have a completely different natural scale. A fair benchmark tunes the learning rate **per optimiser** over a sweep and reports the best of each. This cell does not do that. Say so before the results appear, so the room reads them as an illustration of behaviour rather than as a verdict.

**And prepare the room for the result to be undramatic.** Final test accuracies come in at 98.8%, 99.8%, 99.6% on 500 test points — differences of a handful of examples, from a single seed, with no error bars. **The gaps are not statistically meaningful**, and pretending otherwise would teach a bad habit. What *is* meaningful is the shape of the loss curves: momentum and Adam descend visibly faster in the early epochs, which is the claim the theory actually makes. Convergence speed is the measurable effect; final accuracy on an easy task is not.

**If time allows, ask the question that turns the demo into an experiment.** Rerun the three-way race across five seeds and report mean ± std. It takes a minute of compute and converts an anecdote into a measurement — and it is the discipline that separates a plausible benchmark from a publishable one.
</details>

## 2. The Optimizer Ladder

💡 **Intuition.** **Momentum** treats the gradient as a force, not a velocity: updates accumulate, so consistent directions build speed while zigzag components ([Optimization's](../Intro_Math/Optimization/Optimization.ipynb) κ-canyon!) cancel — a heavy ball rolling through the noise. **Adam** adds a per-parameter yardstick: divide each coordinate's step by its own recent gradient RMS, so rarely-updated weights take bold steps and jittery ones take timid steps — approximate per-axis preconditioning. Neither is magic: they attack curvature and scale imbalance, nothing else.

In [2]:
results = {}
for name, mk in [("SGD lr=0.05", lambda p: torch.optim.SGD(p, lr=0.05)),
                 ("SGD+momentum 0.9", lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),
                 ("Adam lr=1e-3", lambda p: torch.optim.Adam(p, lr=1e-3))]:
    m = make_model(); results[name] = train(m, mk(m.parameters()))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3))
for name, h in results.items():
    axes[0].semilogy(h[:, 0], label=name); axes[1].plot(h[:, 1], label=name)
axes[0].set_title("train loss"); axes[1].set_title("test accuracy")
for ax in axes: ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_xlabel("epoch")
plt.tight_layout(); plt.show()
for name, h in results.items(): print(f"{name:20s} final test acc {h[-1,1]:.1%}")

SGD lr=0.05          final test acc 98.8%
SGD+momentum 0.9     final test acc 99.8%
Adam lr=1e-3         final test acc 99.6%


/tmp/ipykernel_2053618/35627771.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Three optimisers, same architecture, same data, same 150 epochs:

| optimiser | final test acc | misclassified of 500 |
|---|---|---|
| SGD lr=0.05 | 98.8% | 6 |
| SGD + momentum 0.9 | **99.8%** | 1 |
| Adam lr=1e-3 | 99.6% | 2 |

**Read the accuracy column with care, because it is thinner evidence than it looks.** The test set has 500 points, so the whole spread between best and worst is **five examples**. At 99% accuracy the standard error is $\sqrt{p(1-p)/n} \approx 0.44\%$, so a 1-point gap is roughly two standard errors — from a **single seed**, with no repeats and no error bars. The honest statement is that momentum and Adam are *not worse*; "momentum wins" is not supported by this run. Rerun across five seeds and the ranking may well shuffle.

**The loss curves are the real evidence, and they show what the theory predicts.** On the log axis, momentum and Adam descend visibly faster in the early epochs — that is the measurable claim. **Convergence speed is what these optimisers buy; final accuracy on a task this easy is not a place where they can distinguish themselves**, because all three reach essentially the same solution given enough epochs.

**There is also a fairness problem in the setup worth naming, since it is a trap students will meet everywhere.** SGD runs at lr 0.05, Adam at 1e-3. Those numbers are not comparable — Adam divides by a running gradient RMS, so its updates have a completely different natural scale, and its "good" learning rate is typically 10–100× smaller. A fair comparison tunes the learning rate **per optimiser** and reports each at its best. This cell fixes one setting apiece, so it demonstrates *behaviour*, not *superiority*. Most optimiser comparisons in the wild have this flaw; recognising it is the transferable skill.

**What the mechanisms actually are, restated in one line each.** Momentum accumulates the gradient as a force, so consistent directions build a steady-state step $\frac{1}{1-\beta} = 10\times$ larger while zigzag components cancel — the κ-canyon fix from [Optimization](../Intro_Math/Optimization/Optimization.ipynb) S2. Adam divides each coordinate by its own recent gradient RMS, an approximate diagonal preconditioner that equalises axes with very different scales. **Neither is magic and neither knows anything about your problem**; both attack curvature and scale imbalance.

**Which suggests why the effect is muted here.** The spiral is two-dimensional, standardized, and well-conditioned — there is no severe κ and no scale imbalance across coordinates, so there is not much for either method to fix. The place to see the difference is a badly conditioned problem: re-run without the standardization in `spirals`, or on a model with wildly different layer scales, and the gap between plain SGD and the other two opens up sharply.

---
### 🕐 Session 2 of 3 — *Learning-Rate Schedules & Warmup* (~35 min)
**Goal:** big steps to travel, small steps to settle — and why transformers need warmup.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (regularization).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Learning-Rate Schedules & Warmup</b></summary>

**Timing (~35 min).** 10 min why decay is necessary · 10 min cosine · 10 min warmup and why transformers need it · 5 min the honest reading.

**Start from the theorem, not the practice, because decay has a *reason*.** [Optimization](../Intro_Math/Optimization/Optimization.ipynb) S4 proved that SGD with a constant step size does not converge to the optimum — it converges to a **noise ball** whose radius scales with the learning rate. Big steps let you travel; they also prevent you from ever settling. Decay is not a heuristic bolted on by practitioners, it is the only way to close the gap that theorem describes. Students who have seen that plot understand schedules in one sentence.

**Then note what shape the decay should have, and why cosine won.** Anything decreasing satisfies the theory; the empirical question is *how fast*. Step decay drops the rate abruptly and produces visible cliffs in the loss; linear decay is fine; cosine spends longer at high rates early (more exploration) and eases smoothly into a long low-rate tail (careful settling), with no discontinuities for momentum buffers to react to. It is popular because it is smooth, parameter-free once you fix the horizon, and empirically hard to beat — not because it is optimal in any provable sense.

**Warmup deserves the most time, because it is the least intuitive and the most consequential at scale.** At initialisation Adam's second-moment estimates are computed from a handful of noisy batches, so its per-coordinate yardsticks are **garbage** — and dividing by a garbage denominator produces a wild step. A few gentle epochs let those statistics stabilise before real steps are taken. The failure it prevents is not slow convergence but **irrecoverable divergence**: a deep transformer without warmup can take one enormous early step into a region from which it never recovers, and the run is dead within a hundred iterations.

**Make the bias-correction point if the room is strong, since it sharpens the story.** Adam already corrects the $1/(1-\beta^t)$ initialisation bias in its moment estimates, so warmup is not fixing *bias* — it is fixing **variance**. With $\beta_2 = 0.999$ the second moment has an effective window of ~1000 steps, so for the first few hundred steps the RMS estimate is enormously noisy even after debiasing. Warmup buys time for that window to fill.

**Set expectations before the experiment runs, because the result is a null one and the room should not be surprised.** All three schedules land at essentially the same place — final losses 0.0028, 0.0029, 0.0024, and test accuracy 100%, 99.8%, 100%. **The schedules make no difference on this task**, and the notebook says so. Do not manufacture a winner from the third decimal place.

**Explain *why* it is a null result, because that is the actual lesson.** The noise floor that decay exists to fix is proportional to the gradient noise, and here batches of 100 from a clean 1000-point spiral produce very little. No noise floor, nothing for decay to fix. Similarly, a 3-layer MLP has no fragile early-training statistics to protect, so warmup has nothing to rescue. **The intervention is absent because the disease is absent** — which is a much more useful thing for a student to understand than a rigged demo showing cosine winning.

**Close with the practical rule and its justification.** Warmup → cosine is the standard recipe because its cost is zero when it is unnecessary and it saves the run when it is. **Treat schedules as insurance with a zero premium.** Then name the regime where the premium starts paying: large batches, noisy data, deep transformers, long horizons — [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) is where these recipes stop being optional.
</details>

## 3. Schedules

💡 **Intuition.** [Optimization S4](../Intro_Math/Optimization/Optimization.ipynb) proved the constant-step noise floor: finishing requires *decay*. Cosine decay is the popular smooth path from exploration to settlement. **Warmup** (starting tiny and ramping up) protects the fragile early phase — at initialization, curvature estimates (Adam's RMS yardsticks) are garbage computed from a handful of noisy batches; a few gentle epochs let the statistics stabilize before taking real steps. Deep transformers can *diverge irrecoverably* without it.

In [3]:
results2 = {}
for name, lr_lambda in [
    ("constant", lambda ep: 1.0),
    ("cosine decay", lambda ep: 0.5*(1 + np.cos(np.pi*ep/150))),
    ("warmup 10 + cosine", lambda ep: (ep+1)/10 if ep < 10 else 0.5*(1 + np.cos(np.pi*(ep-10)/140))),
]:
    m = make_model()
    opt = torch.optim.Adam(m.parameters(), lr=1e-2)     # deliberately aggressive base LR
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
    results2[name] = train(m, opt, sched=sched)

plt.figure(figsize=(8, 2.8))
for name, h in results2.items(): plt.semilogy(h[:, 0], label=name)
plt.legend(fontsize=8); plt.grid(True, alpha=0.3)
plt.title("same optimizer, aggressive base LR, three schedules"); plt.xlabel("epoch"); plt.ylabel("train loss")
plt.tight_layout(); plt.show()
for name, h in results2.items(): print(f"{name:22s} final loss {h[-1,0]:.4f}  test acc {h[-1,1]:.1%}")

constant               final loss 0.0028  test acc 100.0%
cosine decay           final loss 0.0029  test acc 99.8%
warmup 10 + cosine     final loss 0.0024  test acc 100.0%


/tmp/ipykernel_2053618/2030531181.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Honest reading:** on this small, low-noise task all three schedules land in the same place — the noise floor that decay exists to fix is tiny here (you *saw* it clearly in [Optimization S4](../Intro_Math/Optimization/Optimization.ipynb)'s SGD experiment, where gradient noise was substantial). The ranking flips at scale: large batches of noisy data, deep transformers, and long horizons are where cosine + warmup stop being optional. Treat schedules as insurance whose premium is zero — the standard recipe (warmup → cosine) never hurts and sometimes rescues the run.

---
### 🕐 Session 3 of 3 — *Regularization & Generalization* (~40 min)
**Goal:** weight decay, dropout, early stopping — and a live look at the fit/overfit boundary.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Regularization & Generalization</b></summary>

**Timing (~40 min).** 8 min the memorisation setup · 10 min the three methods · 12 min the experiment and its table · 10 min the honest reading and double descent.

**Explain the experimental design before running it, because the setup is doing the teaching.** We deliberately engineer overfitting: 120 training points, a width-256 network, 800 full-batch epochs, no held-out shuffling. **You cannot study regularisation without first manufacturing the disease** — on the full 1000-point dataset from Session 1 the network generalises fine and every method looks useless. Point at `Xtr[:120]` and say that this line is the experiment.

**Give each method one sentence and one mechanism.** *Weight decay* adds an L2 penalty, shrinking weights every step — and small weights mean small Lipschitz constants, hence smoother functions. *Dropout* silences units at random during training, so no unit can rely on any other and the network is forced into redundant representations; it is an implicit ensemble over exponentially many subnetworks, which connects directly to [Uncertainty in ML](./Uncertainty_in_ML.ipynb). *Early stopping* just quits while validation loss is still falling — regularisation by impatience, and free.

**Flag the AdamW-versus-Adam distinction, since it is a genuine correctness issue and the code quietly gets it right.** With adaptive optimisers, an L2 penalty added to the *loss* gets divided by the per-coordinate RMS along with everything else, so the effective decay differs per parameter and is not the penalty you wrote down. AdamW **decouples** it — decay is applied directly to the weights, outside the adaptive rescaling. That is the entire content of the paper, and it is why the demo uses `AdamW` for the weight-decay arm. Students who use `Adam(weight_decay=...)` and wonder why decay does nothing have met exactly this.

**Teach the train/validation gap as *the* diagnostic, above any particular method.** Both curves falling: keep training. Train falling, validation flat: you are at capacity. Train falling, validation **rising**: memorisation, and you needed to stop already. The plot shows this directly — faint train curves diving while the unregularised solid validation curve turns up. **A student who reads that gap can debug any training run**; one who has memorised three regularisers cannot.

**Then read the table honestly, because the ranking is not what the section title implies.**

| method | best val loss | at epoch | final test acc |
|---|---|---|---|
| none | 0.362 | 52 | 94.4% |
| weight decay 5e-2 | 0.359 | 51 | 93.6% |
| dropout 0.3 | **0.324** | 199 | **96.2%** |

Weight decay is **indistinguishable** from no regularisation here — 0.359 against 0.362, at essentially the same epoch — and its final accuracy is a point *lower*. Only dropout separates. Say this plainly. On 500 test points a 1-point difference is about two standard errors from a single seed, so even dropout's margin is suggestive rather than established.

**The most useful column is the third one, and it deserves the emphasis.** Every method's best validation loss occurs long before epoch 800 — at 52, 51, and 199. **Early stopping alone recovers most of the available benefit, for free, with no hyperparameter to tune.** That is the practical headline of the session, and it is the recommendation to give a room that will go off and train models next week.

**Note what dropout's epoch-199 tells you.** It reaches its best much later, because dropout slows memorisation rather than preventing it — the network needs longer to fit, and longer to overfit. Regularisation and training duration are coupled; a fixed epoch budget silently favours whichever method converges fastest.

**Close with double descent as a stated caveat, not a demonstration.** In modern overparameterised regimes, pushing model size *past* the interpolation threshold can make test error fall again, so the classical U-curve is incomplete. Be clear that this notebook does not demonstrate it. What survives regardless: **monitor the train/validation gap, and when in doubt, more data beats more tricks.**
</details>

## 4. Fighting Memorization

💡 **Intuition.** A big network *can* memorize noise; regularization is a thumb on the scale toward simple functions. **Weight decay** shrinks weights each step (an L2 penalty — small weights ⇒ smoother functions). **Dropout** randomly silences units during training, forcing redundancy — an implicit [ensemble](./Uncertainty_in_ML.ipynb). **Early stopping** just quits while the *validation* loss is still improving — regularization by impatience, and the cheapest of the three. The diagnostic that rules them all: the gap between train and validation curves.

In [4]:
# Overfit on purpose (small noisy data, big net), then regularize three ways
Xs, ys = Xtr[:120], ytr[:120]                      # starve the model
def train_small(model, opt, epochs=800):
    lossf = nn.CrossEntropyLoss(); hist = []
    for ep in range(epochs):
        opt.zero_grad(); loss = lossf(model(Xs), ys); loss.backward(); opt.step()
        with torch.no_grad():
            hist.append((loss.item(), lossf(model(Xte), yte).item(), (model(Xte).argmax(1)==yte).float().mean().item()))
    return np.array(hist)

runs = {}
m = make_model(256); runs["no regularization"] = train_small(m, torch.optim.Adam(m.parameters(), 2e-3))
m = make_model(256); runs["weight decay 5e-2"] = train_small(m, torch.optim.AdamW(m.parameters(), 2e-3, weight_decay=5e-2))
torch.manual_seed(1)
m = nn.Sequential(nn.Linear(2,256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256,256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256,2))
runs["dropout 0.3"] = train_small(m, torch.optim.Adam(m.parameters(), 2e-3))

plt.figure(figsize=(9, 3))
for (name, h), c in zip(runs.items(), ["C0", "C1", "C2"]):
    plt.plot(h[:, 0], c, alpha=0.4)
    plt.plot(h[:, 1], c, label=f"{name} (val)")
plt.legend(fontsize=8); plt.grid(True, alpha=0.3); plt.ylim(0, 1.5)
plt.title("faint = train loss, solid = validation loss: watch the unregularized gap yawn open")
plt.xlabel("epoch"); plt.tight_layout(); plt.show()
for name, h in runs.items():
    best_ep = h[:, 1].argmin()
    print(f"{name:20s} best val loss {h[best_ep,1]:.3f} @ epoch {best_ep} (early stopping would quit here)  final test acc {h[-1,2]:.1%}")

no regularization    best val loss 0.362 @ epoch 52 (early stopping would quit here)  final test acc 94.4%
weight decay 5e-2    best val loss 0.359 @ epoch 51 (early stopping would quit here)  final test acc 93.6%
dropout 0.3          best val loss 0.324 @ epoch 199 (early stopping would quit here)  final test acc 96.2%


/tmp/ipykernel_2053618/3111714112.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xlabel("epoch"); plt.tight_layout(); plt.show()


**Honest reading of the table:** on a 120-sample toy problem the differences are real but modest — dropout helps most here, and early stopping alone recovers most of the benefit for free. Regularization's value *scales with the memorization opportunity*; rerun with `spirals(noise=0.5)` and width 512 to watch the gaps widen.

**On double descent** (stated, worth knowing): in modern regimes, pushing model size *past* the interpolation point can make test error fall *again* — the classical U-curve is incomplete for very large models. The practical takeaways survive: monitor the train/val gap, and when in doubt, more data beats more tricks.

## 5. Conclusion

Momentum cancels zigzag, Adam equalizes axes, schedules trade exploration for settlement, warmup protects fragile statistics, and regularization is a thumb on the scale toward simplicity — each one an experiment you just ran, not a slogan.

---
## Where next

- [Scaling Neural Networks](./Scale_NN/Scale_NN.ipynb) — the systems half of training.
- [Uncertainty in ML](./Uncertainty_in_ML.ipynb) — what the val-gap means for trust.
- [LLMs from the Ground Up](./LLMs_from_the_Ground_Up.ipynb) — these recipes at their most extreme.